# SentinelMCP: LLM-driven sessions (dataset step 7)

Works on **Kaggle** and **Google Colab**. Just run all cells, top to bottom.

1. Turn on a **GPU** (Kaggle: Settings > Accelerator > GPU, and Internet **on**. Colab: Runtime > Change runtime type > T4 GPU).
2. Run all. It builds its own Python 3.12 environment, starts Ollama with `qwen2.5:7b`, writes paraphrases, runs the sessions and prints a summary. Only progress lines are shown; if something fails, the last lines of its log are printed.
3. Results: Kaggle `/kaggle/working/llm_sessions/` (the notebook output); Colab `MyDrive/sentinel_llm_run_full/`.

Everything is fake (secrets, customers, attacks) and only touches the in-process sandbox.

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/kdt523/Sentinel-mcp.git"
COMMIT = "main"            # FINAL run: pin an exact commit hash
PHASE = "all"              # "all" | "paraphrase" | "sessions"
N_BENIGN, N_ATTACK = 270, 180
AGENT_MODEL = "qwen2.5:7b"
FAMILY = '' or None      # top-up run: only this attack family (needs PHASE = "sessions")
FAMILY_N, FAMILY_OFFSET, FAMILY_LIMIT = 30, 0, 30
SESSION_TIMEOUT = 180      # seconds per session before it is cut off
PARAPHRASE_MODEL = "qwen2.5:3b"
SYSTEM = "plain"           # "plain" (naive agent, attacks can succeed) or "hardened"

BASE = "/kaggle/temp" if ON_KAGGLE else "/content"          # scratch space (not saved)
OUT_DIR = "/kaggle/working/llm_sessions" if ON_KAGGLE else "/content/drive/MyDrive/sentinel_llm_run_full"
REPO = f"{BASE}/Sentinel-mcp"
VENV = f"{BASE}/venv"
PY = f"{VENV}/bin/python"
print("environment:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other", "| output:", OUT_DIR)

In [ ]:
import subprocess, collections

def run(cmd, cwd=None, show=None, env=None, tail=40):
    # Run a shell command, print only lines containing a word in `show` (all lines if None).
    # On failure print the last lines of its output and stop.
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, env=env, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    last = collections.deque(maxlen=tail)
    for line in p.stdout:
        line = line.rstrip()
        last.append(line)
        if show is None or any(k in line for k in show):
            print(line, flush=True)
    code = p.wait()
    if code:
        print("--- last lines ---")
        print(chr(10).join(last))
        raise RuntimeError(f"command failed (exit {code}): {cmd[:120]}")

os.makedirs(BASE, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                   shell=True, capture_output=True, text=True)
if g.returncode:
    raise RuntimeError("No GPU found. Kaggle: Settings > Accelerator > GPU. "
                       "Colab: Runtime > Change runtime type > T4 GPU. Then run again.")
print("GPU:", g.stdout.strip())

## 1. Get the code and build a Python 3.12 environment

In [ ]:
run(f"rm -rf {REPO} && git clone -q {REPO_URL} {REPO}", cwd=BASE)
run(f"git checkout -q {COMMIT} && git log --oneline -1", cwd=REPO)
run("pip install -q uv", cwd=BASE)
run(f"uv venv --python 3.12 {VENV}", cwd=BASE)
run(f'uv pip install -q --python {PY} -e ".[agents,tools]" aiosqlite pytest pytest-asyncio', cwd=REPO)
run(f"{PY} -c 'import data.llm_sessions, data.paraphrase, data.llm_stats; print(1)'", show=("1",))
print("environment ready")

## 2. Ollama with the agent model, checked on the GPU

In [ ]:
import json, time, urllib.request
env = {**os.environ, "OLLAMA_KEEP_ALIVE": "2h", "OLLAMA_MODELS": f"{BASE}/ollama_models"}
run("apt-get install -y -qq zstd curl >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1; ollama --version", cwd=BASE, show=("ollama", "version"))
server = subprocess.Popen(["ollama", "serve"], stdout=open(f"{BASE}/ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except OSError:
        time.sleep(1)
else:
    print(open(f"{BASE}/ollama.log").read()[-1500:])
    raise RuntimeError("Ollama did not start")
MODELS = [AGENT_MODEL] + ([PARAPHRASE_MODEL] if PHASE in ("all", "paraphrase") else [])
for m in MODELS:
    run(f"ollama pull {m}", cwd=BASE, show=("success", "rror"))

def ask(model):
    t = time.time()
    req = urllib.request.Request("http://127.0.0.1:11434/api/generate",
        data=json.dumps({"model": model, "prompt": "Say hi in three words.", "stream": False}).encode(),
        headers={"Content-Type": "application/json"})
    out = json.load(urllib.request.urlopen(req, timeout=900))
    print(f"{model}: {time.time() - t:.1f}s -> {out['response'][:50]!r}")

for m in MODELS:
    ask(m)
ps = subprocess.run("ollama ps", shell=True, capture_output=True, text=True).stdout
print(ps)
if "GPU" not in ps:
    raise RuntimeError("Ollama is not using the GPU (see the PROCESSOR column above). "
                       "Check the accelerator setting and run again.")

## 3. Quick self-test of the pipeline (stand-in model, nothing here becomes data)

In [ ]:
r = subprocess.run(f"{PY} -m pytest tests/unit/test_llm_sessions.py tests/unit/test_llm_label.py "
                   "tests/unit/test_paraphrase.py tests/unit/test_llm_stats.py -q -p no:randomly",
                   shell=True, cwd=REPO, capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])

## 4. Paraphrase the benign prompts  (PHASE = all or paraphrase)

In [ ]:
if PHASE in ("all", "paraphrase"):
    run(f"{PY} -m data.paraphrase --model {PARAPHRASE_MODEL} --benign {N_BENIGN} --attack {N_ATTACK}",
        show=("PROGRESS", "accepted", "rror", "Traceback"))
    run(f"cp data/benign/paraphrases.json {OUT_DIR}/paraphrases.json")
    d = json.load(open(f"{REPO}/data/benign/paraphrases.json"))
    print(len(d["prompts"]), "paraphrases accepted,", len(d["rejected"]), "rejected")
    for k, v in list(d["rejected"].items())[:5]:
        print("  rejected", k, "->", v)
else:
    print("skipped (PHASE =", PHASE + ")")

## 5. Run the LLM sessions  (PHASE = all or sessions)

In [ ]:
if PHASE in ("all", "sessions"):
    # The run resumes from what is already on disk, so if it stops for any reason we simply
    # start it again (up to 4 times) instead of losing the whole run.
    for attempt in range(1, 5):
        try:
            extra = f"--timeout {SESSION_TIMEOUT} "
            if FAMILY:
                extra += f"--family {FAMILY} --n {FAMILY_N} --offset {FAMILY_OFFSET} --limit {FAMILY_LIMIT} "
            run(f"{PY} -m data.llm_sessions --benign {N_BENIGN} --attack {N_ATTACK} --model {AGENT_MODEL} "
                f"{extra}--system {SYSTEM} --out {OUT_DIR}/llm_samples.jsonl --log {OUT_DIR}/llm_sessions.jsonl",
                show=("PROGRESS", "ResumeError", "Traceback", "Error"))
            break
        except RuntimeError as exc:
            print(f"attempt {attempt} stopped: {exc}")
            if attempt == 4:
                raise
            print("restarting; finished sessions are skipped")
else:
    print("skipped (PHASE =", PHASE + ")")

## 6. Summary: how often did the attack actually fire?
Attack sessions that did not fire are labelled by what happened and are never counted as gateway catches.

In [ ]:
log = f"{OUT_DIR}/llm_sessions.jsonl"
if os.path.exists(log):
    run(f"{PY} -m data.llm_stats --log {log} --out {OUT_DIR}/llm_stats.json", show=("zzz-print-nothing",))
    st = json.load(open(f"{OUT_DIR}/llm_stats.json"))
    print("sessions:", st["sessions"], "| attack:", st["attack_sessions"], "| benign:", st["benign_sessions"],
          "| errors:", st["errors"], "| mean seconds/session:", st["mean_seconds"])
    a = st["attack_fired"]
    print("attack fired overall:", a["fired"], "/", a["sessions"], f"({a['rate']:.0%})")
    print("benign sessions with a malicious event (should be 0):", st["benign_sessions_with_malicious_event"])
    print()
    for fam, c in st["per_family"].items():
        print(f"  {fam:<30}{c['fired']:>4}/{c['sessions']:<4} {c['rate']:>5.0%}  CI {c['ci95']}{'  (low n)' if c['low_n'] else ''}")
    print()
    print("FLAGS:" if st["flags"] else "no flags")
    for f in st["flags"]:
        print("  -", f)
    run("git rev-parse HEAD", show=("",))
else:
    print("no sessions yet")

In [ ]:
print("Files in", OUT_DIR)
for n in sorted(os.listdir(OUT_DIR)):
    print(f"  {n:<24}{os.path.getsize(os.path.join(OUT_DIR, n)) / 1024:>10.0f} KB")
if ON_COLAB:
    from google.colab import files
    for name in ("paraphrases.json", "llm_sessions.jsonl", "llm_stats.json"):
        if os.path.exists(f"{OUT_DIR}/{name}"):
            files.download(f"{OUT_DIR}/{name}")